# Descriptive Analysis of the Final Dataset

This notebook describes the final movie-level dataset before inferential and predictive modelling.

The descriptive analysis focuses on the corrected retained variables:

- dependent variable: `log_box_office`
- independent variables: `audienceScore`, `initial_combined_sentiment_score`, `log_initial_review_count`

The goal is to understand data quality, distribution shape, spread, and simple relationships before formal modelling.

## Prerequisites and Descriptive Rules

Before descriptive analysis, this notebook checks that:

- the curated dataset exists at `data/final/final.csv`
- the final dataset has the corrected schema
- the selected variables are numeric
- missingness is documented
- raw box office and logged box office are both inspected

This stage is descriptive only. It summarizes patterns; it does not claim causality.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)


def resolve_data_path() -> Path:
    for base in [Path.cwd(), Path.cwd().parent]:
        candidate = base / 'data' / 'final' / 'final.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find data/final/final.csv from the current working directory.')


DATA_PATH = resolve_data_path()
df = pd.read_csv(DATA_PATH)

OUTCOME = 'log_box_office'
RAW_OUTCOME = 'box_office_num'
KEY_PREDICTORS = ['audienceScore', 'initial_combined_sentiment_score', 'log_initial_review_count']
KEY_VARIABLES = [OUTCOME, *KEY_PREDICTORS]
SUPPORTING_COLUMNS = [
    'id',
    'title',
    'boxOffice',
    RAW_OUTCOME,
    'initial_combined_sentiment_label',
    'initial_review_count',
    'initial_positive_review_ratio',
]
REQUIRED_COLUMNS = [*SUPPORTING_COLUMNS, *KEY_VARIABLES]

missing_required = [column for column in REQUIRED_COLUMNS if column not in df.columns]
if missing_required:
    raise KeyError(f'Missing required columns: {missing_required}')

for column in [RAW_OUTCOME, 'initial_review_count', 'initial_positive_review_ratio', *KEY_VARIABLES]:
    df[column] = pd.to_numeric(df[column], errors='coerce')

print(f'Data path: {DATA_PATH}')
print(f'Dataset shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')

## Dataset Structure and Data Quality

This section checks dataset size, duplicate movie identifiers, missing values, and availability of the selected descriptive variables.

In [ ]:
overview = pd.DataFrame({
    'metric': [
        'rows',
        'columns',
        'duplicate_movie_ids',
        'finite_log_box_office',
        'complete_selected_model_rows',
    ],
    'value': [
        len(df),
        len(df.columns),
        int(df['id'].duplicated().sum()),
        int(np.isfinite(df[OUTCOME]).sum()),
        int(df[KEY_VARIABLES].dropna().shape[0]),
    ],
})

missing_summary = pd.DataFrame({
    'non_null_count': df[REQUIRED_COLUMNS].notna().sum(),
    'missing_count': df[REQUIRED_COLUMNS].isna().sum(),
    'missing_pct': df[REQUIRED_COLUMNS].isna().mean() * 100,
}).sort_values('missing_pct', ascending=False)

display(overview)
display(missing_summary)

## Variable Roles

The selected variables are the same variables used to motivate the later inferential model.

In [ ]:
variable_roles = pd.DataFrame([
    ('log_box_office', 'Dependent variable', 'Logged movie box-office performance / popularity'),
    ('audienceScore', 'Independent variable', 'Audience reaction score'),
    ('initial_combined_sentiment_score', 'Independent variable', 'Weighted early-review sentiment score'),
    ('log_initial_review_count', 'Independent variable', 'Logged count of early reviews / early attention'),
    ('box_office_num', 'Supporting variable', 'Raw numeric box office used to justify log transformation'),
    ('initial_combined_sentiment_label', 'Grouping variable', 'Categorical early sentiment label'),
], columns=['variable', 'role', 'meaning'])

display(variable_roles)

## Summary Statistics

The table below summarizes the dependent variable and retained independent variables using central tendency, spread, range, and shape.

In [ ]:
summary_cols = [OUTCOME, *KEY_PREDICTORS]
summary = df[summary_cols].agg(['count', 'mean', 'median', 'std', 'min', 'max']).T
summary['iqr'] = df[summary_cols].quantile(0.75) - df[summary_cols].quantile(0.25)
summary['skew'] = df[summary_cols].skew(numeric_only=True)
summary = summary.round(4)
display(summary)

## Raw Box Office vs Logged Box Office

Raw box office is highly skewed, so `log_box_office` is used as the modelling outcome.

In [ ]:
box_summary = df[[RAW_OUTCOME, OUTCOME]].agg(['count', 'mean', 'median', 'std', 'min', 'max']).T
box_summary['skew'] = df[[RAW_OUTCOME, OUTCOME]].skew(numeric_only=True)
box_summary = box_summary.round(4)
display(box_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(df[RAW_OUTCOME].dropna(), bins=40, kde=True, ax=axes[0], color='#f06a1f')
axes[0].set_title('Raw box office distribution')
axes[0].ticklabel_format(style='plain', axis='x')

sns.histplot(df[OUTCOME].dropna(), bins=35, kde=True, ax=axes[1], color='#3f4aa8')
axes[1].set_title('Logged box office distribution')
plt.tight_layout()
plt.show()

## Univariate Analysis of Retained Variables

Each retained variable is inspected individually using histograms and boxplots.

In [ ]:
plot_vars = [OUTCOME, *KEY_PREDICTORS]
fig, axes = plt.subplots(len(plot_vars), 2, figsize=(14, 4 * len(plot_vars)))
for row, column in enumerate(plot_vars):
    series = df[column].dropna()
    sns.histplot(series, kde=True, ax=axes[row, 0], color='#2a6f97')
    axes[row, 0].set_title(f'{column}: distribution')
    axes[row, 0].set_xlabel(column)

    sns.boxplot(x=series, ax=axes[row, 1], color='#f4a261')
    axes[row, 1].set_title(f'{column}: boxplot')
    axes[row, 1].set_xlabel(column)

plt.tight_layout()
plt.show()

## Correlation Matrix

The correlation matrix shows simple pairwise relationships among the retained variables. It is descriptive and does not control for other predictors.

In [ ]:
corr = df[summary_cols].corr(numeric_only=True).round(4)
display(corr)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.4f', cmap='RdBu_r', center=0, square=True)
plt.title('Correlation matrix of retained variables')
plt.tight_layout()
plt.show()

## Bivariate Analysis with `log_box_office`

These plots compare each retained predictor with the dependent variable.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, predictor in zip(axes, KEY_PREDICTORS):
    plot_df = df[[predictor, OUTCOME]].dropna()
    sns.regplot(
        data=plot_df,
        x=predictor,
        y=OUTCOME,
        scatter_kws={'alpha': 0.25, 's': 18},
        line_kws={'color': '#c1121f'},
        ax=ax,
    )
    ax.set_title(f'{predictor} vs {OUTCOME}')
plt.tight_layout()
plt.show()

## Grouped Descriptive Analysis

Grouped summaries help show how popularity differs by early sentiment label and by early-review-volume quartile.

In [ ]:
by_sentiment = df.groupby('initial_combined_sentiment_label')[[
    OUTCOME,
    'audienceScore',
    'initial_combined_sentiment_score',
    'log_initial_review_count',
]].agg(['count', 'mean', 'median', 'std'])

df['review_count_quartile'] = pd.qcut(
    df['initial_review_count'],
    q=4,
    labels=['Q1 lowest', 'Q2', 'Q3', 'Q4 highest'],
    duplicates='drop',
)

by_review_quartile = df.groupby('review_count_quartile', observed=False)[[
    OUTCOME,
    RAW_OUTCOME,
    'audienceScore',
    'initial_combined_sentiment_score',
]].agg(['count', 'mean', 'median'])

display(by_sentiment)
display(by_review_quartile)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='initial_combined_sentiment_label', y=OUTCOME, ax=axes[0], color='#90be6d')
axes[0].set_title('log_box_office by sentiment label')

sns.boxplot(data=df, x='review_count_quartile', y=OUTCOME, ax=axes[1], color='#577590')
axes[1].set_title('log_box_office by review-count quartile')
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## Descriptive Conclusion

The descriptive evidence shows that:

- `log_box_office` is a more stable popularity outcome than raw box office
- the retained independent variables have meaningful variation
- `log_initial_review_count` has the clearest simple relationship with `log_box_office`
- `audienceScore` is weakly positive with popularity at the simple pairwise level
- early sentiment aligns with audience scores, but not with a simple increase in box office

These findings justify moving into inferential and predictive modelling with the retained variable set.